In [1]:
import pandas as pd
import numpy as np
import os
import time

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    median_absolute_error,
    explained_variance_score
)

print("C8 imports completed successfully!")

C8 imports completed successfully!


In [2]:
energy_fe_path = "../../Feature_engineered_datasets/energy_feature_engineered.csv"
weather_fe_path = "../../Feature_engineered_datasets/weather_feature_engineered.csv"
air_fe_path = "../../Feature_engineered_datasets/air_quality_feature_engineered.csv"

energy_df = pd.read_csv(energy_fe_path)
weather_df = pd.read_csv(weather_fe_path)
air_df = pd.read_csv(air_fe_path)

print("Energy shape:", energy_df.shape)
print("Weather shape:", weather_df.shape)
print("Air Quality shape:", air_df.shape)

print("\nEnergy columns:")
print(energy_df.columns.tolist())

print("\nWeather columns:")
print(weather_df.columns.tolist())

print("\nAir Quality columns:")
print(air_df.columns.tolist())

Energy shape: (1000, 19)
Weather shape: (96429, 18)
Air Quality shape: (500, 19)

Energy columns:
['Timestamp', 'Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'HVACUsage', 'LightingUsage', 'RenewableEnergy', 'DayOfWeek', 'Holiday', 'EnergyConsumption', 'Hour', 'Month', 'Weekend', 'HVAC_Status', 'Lighting_Status', 'Energy_Per_Occupant', 'Renewable_Energy_Ratio', 'Energy_Per_SquareFoot']

Weather columns:
['Formatted Date', 'Summary', 'Precip Type', 'Temperature (C)', 'Apparent Temperature (C)', 'Humidity', 'Wind Speed (km/h)', 'Wind Bearing (degrees)', 'Visibility (km)', 'Pressure (millibars)', 'Daily Summary', 'Year', 'Month', 'Hour', 'DayOfWeek', 'Season', 'Temperature_Difference', 'Wind_Chill_Indicator']

Air Quality columns:
['PM2.5', 'PM10', 'NOx', 'NO2', 'SO2', 'VOCs', 'CO', 'CO2', 'CH4', 'Temperature', 'Humidity', 'Wind_Direction', 'Location_Type', 'Source_Label', 'PM_Ratio', 'NOx_NO2_Ratio', 'CO_CO2_Ratio', 'Pollution_Score', 'Pollution_Level']


In [3]:
target = "EnergyConsumption"

energy_leakage_columns = [
    "Energy_Per_Occupant",
    "Renewable_Energy_Ratio",
    "Energy_Per_SquareFoot"
]

print("Target column:", target)
print("Target present:", target in energy_df.columns)

print("\nLeakage columns:")
for col in energy_leakage_columns:
    print(f"{col}: {'FOUND' if col in energy_df.columns else 'NOT FOUND'}")

print("\nTarget missing values:", energy_df[target].isna().sum())

Target column: EnergyConsumption
Target present: True

Leakage columns:
Energy_Per_Occupant: FOUND
Renewable_Energy_Ratio: FOUND
Energy_Per_SquareFoot: FOUND

Target missing values: 0


In [4]:
energy_features_c8 = [
    "Temperature",
    "Humidity",
    "SquareFootage",
    "Occupancy",
    "HVACUsage",
    "LightingUsage",
    "RenewableEnergy",
    "DayOfWeek",
    "Holiday",
    "Hour",
    "Month",
    "Weekend",
    "HVAC_Status",
    "Lighting_Status"
]

X_energy_c8 = energy_df[energy_features_c8].copy()
y_energy_c8 = energy_df[target].copy()

print("Energy feature matrix shape:", X_energy_c8.shape)
print("Energy target shape:", y_energy_c8.shape)

print("\nEnergy features:")
print(X_energy_c8.columns.tolist())

print("\nMissing values in Energy features:")
print(X_energy_c8.isna().sum().sum())

print("\nMissing values in target:")
print(y_energy_c8.isna().sum())

Energy feature matrix shape: (1000, 14)
Energy target shape: (1000,)

Energy features:
['Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'HVACUsage', 'LightingUsage', 'RenewableEnergy', 'DayOfWeek', 'Holiday', 'Hour', 'Month', 'Weekend', 'HVAC_Status', 'Lighting_Status']

Missing values in Energy features:
0

Missing values in target:
0


In [5]:
categorical_energy_columns_c8 = [
    "HVACUsage",
    "LightingUsage",
    "DayOfWeek",
    "Holiday",
    "Weekend"
]

X_energy_c8_encoded = pd.get_dummies(
    X_energy_c8,
    columns=categorical_energy_columns_c8,
    drop_first=True,
    dtype=int
)

print("Encoded Energy feature matrix shape:", X_energy_c8_encoded.shape)

print("\nEncoded columns:")
print(X_energy_c8_encoded.columns.tolist())

print("\nMissing values:")
print(X_energy_c8_encoded.isna().sum().sum())

Encoded Energy feature matrix shape: (1000, 19)

Encoded columns:
['Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'RenewableEnergy', 'Hour', 'Month', 'HVAC_Status', 'Lighting_Status', 'HVACUsage_On', 'LightingUsage_On', 'DayOfWeek_Monday', 'DayOfWeek_Saturday', 'DayOfWeek_Sunday', 'DayOfWeek_Thursday', 'DayOfWeek_Tuesday', 'DayOfWeek_Wednesday', 'Holiday_Yes', 'Weekend_Yes']

Missing values:
0


In [6]:
weather_context_features_c8 = [
    "Temperature (C)",
    "Apparent Temperature (C)",
    "Humidity",
    "Wind Speed (km/h)",
    "Visibility (km)",
    "Pressure (millibars)"
]

weather_context_c8 = weather_df[
    ["Month", "Hour"] + weather_context_features_c8
].copy()

print("Weather context shape:", weather_context_c8.shape)

print("\nWeather context columns:")
print(weather_context_c8.columns.tolist())

print("\nMissing values:")
print(weather_context_c8.isna().sum())

Weather context shape: (96429, 8)

Weather context columns:
['Month', 'Hour', 'Temperature (C)', 'Apparent Temperature (C)', 'Humidity', 'Wind Speed (km/h)', 'Visibility (km)', 'Pressure (millibars)']

Missing values:
Month                       0
Hour                        0
Temperature (C)             0
Apparent Temperature (C)    0
Humidity                    0
Wind Speed (km/h)           0
Visibility (km)             0
Pressure (millibars)        0
dtype: int64


In [7]:
weather_context_avg_c8 = (
    weather_context_c8
    .groupby(["Month", "Hour"])[weather_context_features_c8]
    .mean()
    .reset_index()
)

print("Aggregated Weather context shape:", weather_context_avg_c8.shape)

print("\nNumber of Month-Hour groups:", len(weather_context_avg_c8))

print("\nMissing values:")
print(weather_context_avg_c8.isna().sum())

print("\nFirst 10 Month-Hour groups:")
display(weather_context_avg_c8.head(10))

Aggregated Weather context shape: (288, 8)

Number of Month-Hour groups: 288

Missing values:
Month                       0
Hour                        0
Temperature (C)             0
Apparent Temperature (C)    0
Humidity                    0
Wind Speed (km/h)           0
Visibility (km)             0
Pressure (millibars)        0
dtype: int64

First 10 Month-Hour groups:


,Month,Hour,Temperature (C),Apparent Temperature (C),Humidity,Wind Speed (km/h),Visibility (km),Pressure (millibars)
0,1,0,-0.291105,-2.963718,0.883607,10.346729,8.064353,1019.761290
1,1,1,-0.410427,-3.252786,0.881056,10.985346,8.196175,1001.607302
2,1,2,-0.566846,-3.283399,0.883959,10.520854,8.157979,1007.598886
3,1,3,-0.614614,-3.371717,0.889795,10.311224,7.926346,1019.522669
4,1,4,-0.772988,-3.598110,0.886041,10.777793,6.872859,998.397713
5,1,5,-0.790893,-3.610101,0.887947,10.718917,6.775126,1007.400088
6,1,6,-0.761518,-3.637341,0.890528,10.695924,6.348830,1019.668739
7,1,7,-0.457592,-3.307739,0.884839,11.017641,6.405959,992.829912
8,1,8,0.391544,-2.569762,0.864516,11.841385,6.765494,990.096246
9,1,9,1.311274,-1.705279,0.841173,12.672447,7.307559,1020.303578


In [8]:
energy_weather_c8 = energy_df.merge(
    weather_context_avg_c8,
    on=["Month", "Hour"],
    how="left",
    suffixes=("", "_Weather")
)

print("Merged Energy + Weather shape:", energy_weather_c8.shape)

print("\nColumns:")
print(energy_weather_c8.columns.tolist())

print("\nMissing values in Weather context:")
print(
    energy_weather_c8[weather_context_features_c8]
    .isna()
    .sum()
)

print("\nTotal missing values:")
print(energy_weather_c8.isna().sum().sum())

Merged Energy + Weather shape: (1000, 25)

Columns:
['Timestamp', 'Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'HVACUsage', 'LightingUsage', 'RenewableEnergy', 'DayOfWeek', 'Holiday', 'EnergyConsumption', 'Hour', 'Month', 'Weekend', 'HVAC_Status', 'Lighting_Status', 'Energy_Per_Occupant', 'Renewable_Energy_Ratio', 'Energy_Per_SquareFoot', 'Temperature (C)', 'Apparent Temperature (C)', 'Humidity_Weather', 'Wind Speed (km/h)', 'Visibility (km)', 'Pressure (millibars)']

Missing values in Weather context:
Temperature (C)             0
Apparent Temperature (C)    0
Humidity                    0
Wind Speed (km/h)           0
Visibility (km)             0
Pressure (millibars)        0
dtype: int64

Total missing values:
0


In [9]:
air_context_features_c8 = [
    "PM2.5",
    "PM10",
    "NOx",
    "NO2",
    "SO2",
    "VOCs",
    "CO",
    "CO2",
    "CH4",
    "Temperature",
    "Humidity",
    "Wind_Direction"
]

air_context_stats_c8 = air_df[air_context_features_c8].agg(
    ["mean", "std"]
)

print("Air Quality context statistics:")
display(air_context_stats_c8)

print("\nMissing values:")
print(air_df[air_context_features_c8].isna().sum())

Air Quality context statistics:


,PM2.5,PM10,NOx,NO2,SO2,VOCs,CO,CO2,CH4,Temperature,Humidity,Wind_Direction
mean,54.671620,84.703251,92.005278,42.493194,11.542440,112.193098,2.177705,439.288087,2.130439,30.448413,56.196914,180.842000
std,22.799473,30.567912,54.653744,26.268991,7.267972,39.471686,0.824762,51.220578,0.391396,2.957290,11.035563,102.323767



Missing values:
PM2.5             0
PM10              0
NOx               0
NO2               0
SO2               0
VOCs              0
CO                0
CO2               0
CH4               0
Temperature       0
Humidity          0
Wind_Direction    0
dtype: int64


In [10]:
air_context_means_c8 = (
    air_df[air_context_features_c8]
    .mean()
    .to_dict()
)

print("Air Quality context features:")
for feature, value in air_context_means_c8.items():
    print(f"{feature}: {value:.6f}")

# Add Air Quality context to every Energy-Weather record
for feature, value in air_context_means_c8.items():
    energy_weather_c8[f"Air_{feature}_Mean"] = value

print("\nUpdated dataset shape:", energy_weather_c8.shape)

print("\nNew Air Quality context columns:")
print([
    col for col in energy_weather_c8.columns
    if col.startswith("Air_")
])

Air Quality context features:
PM2.5: 54.671620
PM10: 84.703251
NOx: 92.005278
NO2: 42.493194
SO2: 11.542440
VOCs: 112.193098
CO: 2.177705
CO2: 439.288087
CH4: 2.130439
Temperature: 30.448413
Humidity: 56.196914
Wind_Direction: 180.842000

Updated dataset shape: (1000, 37)

New Air Quality context columns:
['Air_PM2.5_Mean', 'Air_PM10_Mean', 'Air_NOx_Mean', 'Air_NO2_Mean', 'Air_SO2_Mean', 'Air_VOCs_Mean', 'Air_CO_Mean', 'Air_CO2_Mean', 'Air_CH4_Mean', 'Air_Temperature_Mean', 'Air_Humidity_Mean', 'Air_Wind_Direction_Mean']


In [11]:
energy_weather_c8["Weather_Energy_Temp_Difference"] = (
    energy_weather_c8["Temperature (C)"]
    - energy_weather_c8["Temperature"]
)

energy_weather_c8["Weather_Energy_Humidity_Difference"] = (
    energy_weather_c8["Humidity_Weather"]
    - energy_weather_c8["Humidity"]
)

energy_weather_c8["Occupancy_Density"] = (
    energy_weather_c8["Occupancy"]
    / (energy_weather_c8["SquareFootage"] + 1e-6)
)

interaction_features_c8 = [
    "Weather_Energy_Temp_Difference",
    "Weather_Energy_Humidity_Difference",
    "Occupancy_Density"
]

print("Interaction features created:")
print(interaction_features_c8)

print("\nMissing values:")
print(
    energy_weather_c8[interaction_features_c8]
    .isna()
    .sum()
)

print("\nInteraction feature statistics:")
display(
    energy_weather_c8[interaction_features_c8].describe()
)

Interaction features created:
['Weather_Energy_Temp_Difference', 'Weather_Energy_Humidity_Difference', 'Occupancy_Density']

Missing values:
Weather_Energy_Temp_Difference        0
Weather_Energy_Humidity_Difference    0
Occupancy_Density                     0
dtype: int64

Interaction feature statistics:


,Weather_Energy_Temp_Difference,Weather_Energy_Humidity_Difference,Occupancy_Density
count,1000.000000,1000.000000,1000.000000
mean,-23.819567,-44.554390,0.003171
std,3.280384,8.517585,0.002108
min,-30.718398,-59.146489,0.000000
25%,-26.274141,-51.547936,0.001475
50%,-23.636125,-45.130802,0.003112
75%,-21.494904,-37.541926,0.004783
max,-14.747672,-29.125447,0.008684


In [12]:
energy_features_for_c8 = [
    "Temperature",
    "Humidity",
    "SquareFootage",
    "Occupancy",
    "HVACUsage",
    "LightingUsage",
    "RenewableEnergy",
    "DayOfWeek",
    "Holiday",
    "Hour",
    "Month",
    "Weekend",
    "HVAC_Status",
    "Lighting_Status"
]

weather_features_for_c8 = [
    "Temperature (C)",
    "Apparent Temperature (C)",
    "Humidity_Weather",
    "Wind Speed (km/h)",
    "Visibility (km)",
    "Pressure (millibars)"
]

air_features_for_c8 = [
    "Air_PM2.5_Mean",
    "Air_PM10_Mean",
    "Air_NOx_Mean",
    "Air_NO2_Mean",
    "Air_SO2_Mean",
    "Air_VOCs_Mean",
    "Air_CO_Mean",
    "Air_CO2_Mean",
    "Air_CH4_Mean",
    "Air_Temperature_Mean",
    "Air_Humidity_Mean",
    "Air_Wind_Direction_Mean"
]

X_c8 = energy_weather_c8[
    energy_features_for_c8
    + weather_features_for_c8
    + interaction_features_c8
    + air_features_for_c8
].copy()

y_c8 = energy_weather_c8[target].copy()

print("C8 feature matrix shape:", X_c8.shape)
print("C8 target shape:", y_c8.shape)

print("\nNumber of input features:", X_c8.shape[1])

print("\nMissing values in X:")
print(X_c8.isna().sum().sum())

print("\nMissing values in y:")
print(y_c8.isna().sum())

print("\nFeature groups:")
print("Energy features:", len(energy_features_for_c8))
print("Weather features:", len(weather_features_for_c8))
print("Interaction features:", len(interaction_features_c8))
print("Air Quality features:", len(air_features_for_c8))

C8 feature matrix shape: (1000, 35)
C8 target shape: (1000,)

Number of input features: 35

Missing values in X:
0

Missing values in y:
0

Feature groups:
Energy features: 14
Weather features: 6
Interaction features: 3
Air Quality features: 12


In [13]:
categorical_fused_columns_c8 = [
    "HVACUsage",
    "LightingUsage",
    "DayOfWeek",
    "Holiday",
    "Weekend"
]

X_c8_encoded = pd.get_dummies(
    X_c8,
    columns=categorical_fused_columns_c8,
    drop_first=True,
    dtype=int
)

print("Encoded C8 feature matrix shape:", X_c8_encoded.shape)

print("\nMissing values:")
print(X_c8_encoded.isna().sum().sum())

print("\nData types:")
print(X_c8_encoded.dtypes.value_counts())

print("\nEncoded feature columns:")
print(X_c8_encoded.columns.tolist())

Encoded C8 feature matrix shape: (1000, 40)

Missing values:
0

Data types:
float64    25
int64      15
Name: count, dtype: int64

Encoded feature columns:
['Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'RenewableEnergy', 'Hour', 'Month', 'HVAC_Status', 'Lighting_Status', 'Temperature (C)', 'Apparent Temperature (C)', 'Humidity_Weather', 'Wind Speed (km/h)', 'Visibility (km)', 'Pressure (millibars)', 'Weather_Energy_Temp_Difference', 'Weather_Energy_Humidity_Difference', 'Occupancy_Density', 'Air_PM2.5_Mean', 'Air_PM10_Mean', 'Air_NOx_Mean', 'Air_NO2_Mean', 'Air_SO2_Mean', 'Air_VOCs_Mean', 'Air_CO_Mean', 'Air_CO2_Mean', 'Air_CH4_Mean', 'Air_Temperature_Mean', 'Air_Humidity_Mean', 'Air_Wind_Direction_Mean', 'HVACUsage_On', 'LightingUsage_On', 'DayOfWeek_Monday', 'DayOfWeek_Saturday', 'DayOfWeek_Sunday', 'DayOfWeek_Thursday', 'DayOfWeek_Tuesday', 'DayOfWeek_Wednesday', 'Holiday_Yes', 'Weekend_Yes']


In [14]:
X_train_c8, X_test_c8, y_train_c8, y_test_c8 = train_test_split(
    X_c8_encoded,
    y_c8,
    test_size=0.20,
    random_state=42
)

print("Training feature shape:", X_train_c8.shape)
print("Testing feature shape:", X_test_c8.shape)

print("\nTraining target shape:", y_train_c8.shape)
print("Testing target shape:", y_test_c8.shape)

print("\nTraining target mean:", y_train_c8.mean())
print("Testing target mean:", y_test_c8.mean())

Training feature shape: (800, 40)
Testing feature shape: (200, 40)

Training target shape: (800,)
Testing target shape: (200,)

Training target mean: 77.04935351116563
Testing target mean: 77.08195029880147


In [15]:
start_time_c8_lr = time.time()

c8_lr = LinearRegression()

c8_lr.fit(X_train_c8, y_train_c8)

y_pred_c8_lr = c8_lr.predict(X_test_c8)

c8_lr_time = time.time() - start_time_c8_lr

c8_lr_mae = mean_absolute_error(y_test_c8, y_pred_c8_lr)
c8_lr_mse = mean_squared_error(y_test_c8, y_pred_c8_lr)
c8_lr_rmse = np.sqrt(c8_lr_mse)
c8_lr_r2 = r2_score(y_test_c8, y_pred_c8_lr)

c8_lr_smape = (
    100
    * np.mean(
        2 * np.abs(y_test_c8 - y_pred_c8_lr)
        / (np.abs(y_test_c8) + np.abs(y_pred_c8_lr) + 1e-8)
    )
)

c8_lr_nrmse = (
    c8_lr_rmse / np.mean(y_test_c8)
) * 100

c8_lr_medae = median_absolute_error(y_test_c8, y_pred_c8_lr)
c8_lr_ev = explained_variance_score(y_test_c8, y_pred_c8_lr)

print("C8 Three-Domain Linear Regression")
print("-" * 45)
print(f"MAE:                {c8_lr_mae:.4f}")
print(f"MSE:                {c8_lr_mse:.4f}")
print(f"RMSE:               {c8_lr_rmse:.4f}")
print(f"R²:                 {c8_lr_r2:.4f}")
print(f"sMAPE:              {c8_lr_smape:.4f}%")
print(f"NRMSE:              {c8_lr_nrmse:.4f}%")
print(f"Median Absolute Error: {c8_lr_medae:.4f}")
print(f"Explained Variance: {c8_lr_ev:.4f}")
print(f"Training Time:      {c8_lr_time:.4f} seconds")

C8 Three-Domain Linear Regression
---------------------------------------------
MAE:                4.2218
MSE:                27.9129
RMSE:               5.2833
R²:                 0.5738
sMAPE:              5.5399%
NRMSE:              6.8541%
Median Absolute Error: 3.6630
Explained Variance: 0.5739
Training Time:      0.0229 seconds


In [16]:
start_time_c8_rf = time.time()

c8_rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

c8_rf.fit(X_train_c8, y_train_c8)

y_pred_c8_rf = c8_rf.predict(X_test_c8)

c8_rf_time = time.time() - start_time_c8_rf

c8_rf_mae = mean_absolute_error(y_test_c8, y_pred_c8_rf)
c8_rf_mse = mean_squared_error(y_test_c8, y_pred_c8_rf)
c8_rf_rmse = np.sqrt(c8_rf_mse)
c8_rf_r2 = r2_score(y_test_c8, y_pred_c8_rf)

c8_rf_smape = (
    100
    * np.mean(
        2 * np.abs(y_test_c8 - y_pred_c8_rf)
        / (np.abs(y_test_c8) + np.abs(y_pred_c8_rf) + 1e-8)
    )
)

c8_rf_nrmse = (
    c8_rf_rmse / np.mean(y_test_c8)
) * 100

c8_rf_medae = median_absolute_error(y_test_c8, y_pred_c8_rf)
c8_rf_ev = explained_variance_score(y_test_c8, y_pred_c8_rf)

print("C8 Three-Domain Random Forest")
print("-" * 45)
print(f"MAE:                   {c8_rf_mae:.4f}")
print(f"MSE:                   {c8_rf_mse:.4f}")
print(f"RMSE:                  {c8_rf_rmse:.4f}")
print(f"R²:                    {c8_rf_r2:.4f}")
print(f"sMAPE:                 {c8_rf_smape:.4f}%")
print(f"NRMSE:                 {c8_rf_nrmse:.4f}%")
print(f"Median Absolute Error: {c8_rf_medae:.4f}")
print(f"Explained Variance:    {c8_rf_ev:.4f}")
print(f"Training Time:         {c8_rf_time:.4f} seconds")

C8 Three-Domain Random Forest
---------------------------------------------
MAE:                   4.3971
MSE:                   29.9566
RMSE:                  5.4733
R²:                    0.5426
sMAPE:                 5.7482%
NRMSE:                 7.1006%
Median Absolute Error: 4.0448
Explained Variance:    0.5427
Training Time:         0.2836 seconds


In [17]:
# C8 Step 17 — Validation-Based LR + RF Hybrid

X_train_inner_c8, X_val_c8, y_train_inner_c8, y_val_c8 = train_test_split(
    X_train_c8,
    y_train_c8,
    test_size=0.20,
    random_state=42
)

# Train LR
c8_lr_val = LinearRegression()
c8_lr_val.fit(X_train_inner_c8, y_train_inner_c8)

# Train RF
c8_rf_val = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
c8_rf_val.fit(X_train_inner_c8, y_train_inner_c8)

# Validation predictions
y_val_lr_c8 = c8_lr_val.predict(X_val_c8)
y_val_rf_c8 = c8_rf_val.predict(X_val_c8)

# Test different weights
weights_c8 = np.arange(0, 1.01, 0.05)

best_weight_c8 = None
best_rmse_c8 = float("inf")

for w in weights_c8:

    y_val_hybrid_c8 = (
        w * y_val_lr_c8
        + (1 - w) * y_val_rf_c8
    )

    rmse = np.sqrt(
        mean_squared_error(y_val_c8, y_val_hybrid_c8)
    )

    if rmse < best_rmse_c8:
        best_rmse_c8 = rmse
        best_weight_c8 = w

best_rf_weight_c8 = 1 - best_weight_c8

print("C8 Validation-Based Hybrid")
print("-" * 45)
print(f"Best LR Weight: {best_weight_c8:.2f}")
print(f"Best RF Weight: {best_rf_weight_c8:.2f}")
print(f"Validation RMSE: {best_rmse_c8:.4f}")

C8 Validation-Based Hybrid
---------------------------------------------
Best LR Weight: 0.60
Best RF Weight: 0.40
Validation RMSE: 5.4402


In [18]:
# C8 Step 18 — Final Three-Domain Hybrid Evaluation

# Retrain both models on the complete training data
start_time_c8_hybrid = time.time()

c8_lr_final = LinearRegression()
c8_lr_final.fit(X_train_c8, y_train_c8)

c8_rf_final = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
c8_rf_final.fit(X_train_c8, y_train_c8)

# Predictions on untouched test set
y_pred_c8_lr_final = c8_lr_final.predict(X_test_c8)
y_pred_c8_rf_final = c8_rf_final.predict(X_test_c8)

# Validation-selected weights
lr_weight_c8 = best_weight_c8
rf_weight_c8 = best_rf_weight_c8

# Final hybrid prediction
y_pred_c8_hybrid = (
    lr_weight_c8 * y_pred_c8_lr_final
    + rf_weight_c8 * y_pred_c8_rf_final
)

c8_hybrid_time = time.time() - start_time_c8_hybrid

# Metrics
c8_hybrid_mae = mean_absolute_error(
    y_test_c8,
    y_pred_c8_hybrid
)

c8_hybrid_mse = mean_squared_error(
    y_test_c8,
    y_pred_c8_hybrid
)

c8_hybrid_rmse = np.sqrt(c8_hybrid_mse)

c8_hybrid_r2 = r2_score(
    y_test_c8,
    y_pred_c8_hybrid
)

c8_hybrid_smape = (
    100
    * np.mean(
        2 * np.abs(y_test_c8 - y_pred_c8_hybrid)
        / (
            np.abs(y_test_c8)
            + np.abs(y_pred_c8_hybrid)
            + 1e-8
        )
    )
)

c8_hybrid_nrmse = (
    c8_hybrid_rmse / np.mean(y_test_c8)
) * 100

c8_hybrid_medae = median_absolute_error(
    y_test_c8,
    y_pred_c8_hybrid
)

c8_hybrid_ev = explained_variance_score(
    y_test_c8,
    y_pred_c8_hybrid
)

print("C8 Three-Domain Hybrid")
print("-" * 45)
print(f"LR Weight:             {lr_weight_c8:.2f}")
print(f"RF Weight:             {rf_weight_c8:.2f}")
print(f"MAE:                   {c8_hybrid_mae:.4f}")
print(f"MSE:                   {c8_hybrid_mse:.4f}")
print(f"RMSE:                  {c8_hybrid_rmse:.4f}")
print(f"R²:                    {c8_hybrid_r2:.4f}")
print(f"sMAPE:                 {c8_hybrid_smape:.4f}%")
print(f"NRMSE:                 {c8_hybrid_nrmse:.4f}%")
print(f"Median Absolute Error: {c8_hybrid_medae:.4f}")
print(f"Explained Variance:    {c8_hybrid_ev:.4f}")
print(f"Training Time:         {c8_hybrid_time:.4f} seconds")

C8 Three-Domain Hybrid
---------------------------------------------
LR Weight:             0.60
RF Weight:             0.40
MAE:                   4.2262
MSE:                   27.8657
RMSE:                  5.2788
R²:                    0.5746
sMAPE:                 5.5368%
NRMSE:                 6.8483%
Median Absolute Error: 3.8290
Explained Variance:    0.5746
Training Time:         0.2962 seconds
